# Импорты

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import csv
import re
from pathlib import Path

import torch
import pytorch_lightning as pl
from omegaconf import OmegaConf
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

from src.training.gqa.lightning_module import GQALightningModule
from src.data.wikitext_datamodule import WikiTextDataModule
from src.inference.gqa_cached_generator import GQACachedGenerator
from src.tokenization.tokenizers import load_bpe_tokenizer

import warnings
warnings.filterwarnings("ignore", message=".*LeafSpec.*")

import omegaconf
torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])

W0919 01:44:22.085000 23804 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# ClearML

In [ ]:
from clearml import Task
from dotenv import load_dotenv
load_dotenv()

task = Task.init(project_name="GQA-Language-Model", task_name="GQA_Training")

ClearML Task: overwriting (reusing) task id=b5854f5234d04023b19824f29c912e58
ClearML results page: https://app.clear.ml/projects/959a04b3fe7845d7a881a3e957ece073/tasks/b5854f5234d04023b19824f29c912e58/output/log


# Конфиг

In [ ]:
config = OmegaConf.load("../configs/gqa_model_config.yaml")

# print(OmegaConf.to_yaml(config))  # удобно сразу видеть, с чем запускаемся
task.connect(OmegaConf.to_container(config, resolve=True))

{'model': {'vocab_size': 10000,
  'd_model': 512,
  'n_heads': 8,
  'n_kv_heads': 2,
  'n_layers': 6,
  'd_ff': 2048,
  'max_len': 512},
 'training': {'batch_size': 16,
  'learning_rate': 0.0003,
  'weight_decay': 0.1,
  'warmup_steps': 1000,
  'max_epochs': 3,
  'gradient_clip_val': 1.0,
  'optimizer': {'name': 'adamw', 'betas': [0.9, 0.999], 'eps': 1e-08},
  'scheduler': {'name': 'cosine', 'T_max': 52044, 'eta_min': 1e-06}},
 'data': {'max_length': 512, 'batch_size': 16, 'num_workers': 0},
 'paths': {'data_dir': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/data/processed/wikitext/stage2_filtered',
  'tokenizer_path': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/data/tokenizers/bpe_tokenizer.json',
  'checkpoint_dir': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/checkpoints/gqa'}}

# Модель и датамодуль

In [ ]:
model = GQALightningModule(config)
datamodule = WikiTextDataModule(config)

# Полезная проверка перед долгим обучением — сколько параметров в модели
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

Total parameters: 26,800,400


# Callbacks и logger

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=config.paths.checkpoint_dir,   # ${oc.env:CHECKPOINT_DIR}/gqa
    filename="gqa-{epoch:02d}-{val_perplexity:.2f}",
    monitor="val_perplexity",
    mode="min",
    save_top_k=3,
    save_last=True,
)


class NamedModelCheckpoint(ModelCheckpoint):
    """
    ModelCheckpoint.state_key строится только из monitor/mode/every_n_*
    (filename и dirpath в него не входят), поэтому два ModelCheckpoint
    с одинаковыми monitor="val_perplexity"/mode="min" получают одинаковый
    state_key, и Lightning отказывается их запускать вместе
    (RuntimeError: Found more than one stateful callback of type
    ModelCheckpoint). Различаем их явным именем в state_key.
    """
    def __init__(self, *args, state_key_name: str, **kwargs):
        super().__init__(*args, **kwargs)
        self._state_key_name = state_key_name

    @property
    def state_key(self) -> str:
        return f"{super().state_key}[{self._state_key_name}]"


# второй ModelCheckpoint с фиксированным именем — всегда указывает
# на текущую лучшую модель, без разбора чисел в имени файла
best_checkpoint_callback = NamedModelCheckpoint(
    state_key_name="best_fixed_name",
    dirpath=config.paths.checkpoint_dir,
    filename="best",
    monitor="val_perplexity",
    mode="min",
    save_top_k=1,
    save_last=False,
    verbose=True,
)

lr_monitor = LearningRateMonitor(logging_interval="step")
tb_logger = TensorBoardLogger(save_dir="../logs/", name="gqa_training")


class CheckpointMetricsRecorder(pl.Callback):
    """
    Пишет манифест checkpoints/gqa/checkpoints_metrics.csv — по строке
    на каждый чекпоинт, реально оставшийся на диске (сверяется с
    checkpoint_callback.best_k_models, чтобы не описывать файлы,
    удалённые top-k ротацией). Определён в ноутбуке — не новый .py модуль.
    """
    def __init__(self, checkpoint_callback, manifest_path):
        self.checkpoint_callback = checkpoint_callback
        self.manifest_path = Path(manifest_path)
        self.metrics_by_epoch = {}

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            # Trainer.fit() по умолчанию прогоняет пару val-батчей до старта
            # обучения (num_sanity_val_steps) — train_loss/train_perplexity
            # в это время ещё не залогированы ни разу.
            return
        m = trainer.callback_metrics
        self.metrics_by_epoch[trainer.current_epoch] = {
            "epoch": trainer.current_epoch,
            "train_loss": float(m.get("train_loss_epoch", m.get("train_loss"))),
            "val_loss": float(m["val_loss"]),
            "train_perplexity": float(m.get("train_perplexity_epoch", m.get("train_perplexity"))),
            "val_perplexity": float(m["val_perplexity"]),
        }

    def save(self):
        paths = list(self.checkpoint_callback.best_k_models.keys())
        if self.checkpoint_callback.last_model_path:
            paths.append(self.checkpoint_callback.last_model_path)
        with open(self.manifest_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "checkpoint", "epoch", "train_loss", "val_loss",
                "train_perplexity", "val_perplexity",
            ])
            writer.writeheader()
            for path in paths:
                # Имена вида "gqa-epoch=00-val_perplexity=23.20.ckpt" — номер
                # эпохи ищем по regex, а не split("-")[1] (там лежит "epoch=00",
                # а не голое число). "best.ckpt"/"last.ckpt" совпадения не дают.
                match = re.search(r"epoch=(\d+)", Path(path).stem)
                epoch = int(match.group(1)) if match else None
                writer.writerow({"checkpoint": Path(path).name,
                                  **self.metrics_by_epoch.get(epoch, {})})


metrics_recorder = CheckpointMetricsRecorder(
    checkpoint_callback=checkpoint_callback,
    manifest_path=Path(config.paths.checkpoint_dir) / "checkpoints_metrics.csv",
)

# Sanity check

In [ ]:
sanity_trainer = pl.Trainer(
    fast_dev_run=True,   # прогоняет 1 train + 1 val батч и останавливается
    accelerator="gpu",
    devices=1,
)
sanity_trainer.fit(model, datamodule=datamodule)
print("Sanity check passed ✓")

# Обучение

In [ ]:
trainer = pl.Trainer(
    max_epochs=config.training.max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    gradient_clip_val=config.training.gradient_clip_val,
    gradient_clip_algorithm="norm",
    callbacks=[checkpoint_callback, best_checkpoint_callback, lr_monitor, metrics_recorder],
    logger=tb_logger,
    log_every_n_steps=10,
)
trainer.fit(model, datamodule=datamodule)
metrics_recorder.save()

# Отчёт по DoD

In [11]:
best_val_perplexity = checkpoint_callback.best_model_score.item()
print(f"Best val perplexity: {best_val_perplexity:.2f}")
if best_val_perplexity <= 25:
    print("Критерий ≤25 выполнен: 2 + 3 = 5 баллов")
elif best_val_perplexity <= 40:
    print("Критерий ≤40 выполнен: 2 балла (порог ≤25 не достигнут)")
else:
    print(f"Ни один порог не достигнут (нужно ≤40, получено {best_val_perplexity:.2f})")

print(f"Best checkpoint (fixed name): {best_checkpoint_callback.best_model_path}")
print(f"Checkpoints metrics manifest: {metrics_recorder.manifest_path}")

AttributeError: 'NoneType' object has no attribute 'item'

ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


# Завершение ClearML Task

In [ ]:
task.close()

# Генерация текста (п.1.4): инициализация из чекпоинта

In [12]:
# "Не забудьте реализовать инициализацию обученной модели из сохранённых весов" —
# берём best.ckpt из п.1.3 (фиксированное имя, однозначно определяет лучшую модель).
generator = GQACachedGenerator.from_checkpoint("../checkpoints/gqa/best.ckpt")
print("Модель восстановлена из checkpoints/gqa/best.ckpt")
print(f"d_model={generator.model.d_model}, n_layers={len(generator.model.transformer_layers)}")

Модель восстановлена из checkpoints/gqa/best.ckpt
d_model=512, n_layers=6


# Токенизатор

In [13]:
load_dotenv()

# Путь к токенизатору берём из hparams, сохранённых в самом чекпоинте —
# отдельно читать gqa_model_config.yaml не нужно.
tokenizer = load_bpe_tokenizer(generator.lightning_module.hparams.paths.tokenizer_path)
print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")

Tokenizer vocab size: 10000


# Генерация — несколько промптов

In [ ]:
prompts = [
    "The history of",
    "In the beginning",
    "The city is",
    "Scientists have discovered",
    "Pornhub is",
]

for prompt in prompts:
    generated_text = generator.generate(
        prompt,
        tokenizer,
        max_length=100,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )
    print(f"Prompt: {prompt!r}")
    print(f"Generated: {generated_text}")
    print("-" * 60)

Prompt: 'The history of'
Generated: The history of the fleet has been extensively reported in the British Empire . However , many other contingents of the fleet have suggested that the fleet may have been a permanent part of the fleet , and they have been subject to conflict with the British Admiral Hipper in 1837 . <eos>
------------------------------------------------------------
Prompt: 'In the beginning'
Generated: In the beginning of the 20th century , the town was constructed by a merger of the Cheshire Cheshire ' s Henry Sheriff and St . John ' s Church . The town became a commercial center for the town in 1908 . The town was home to many local people , including the local community , who was involved in the development of the town . The town ' s residence has been established by the city ' s residents . <eos>
------------------------------------------------------------
Prompt: 'The city is'
Generated: The city is the center of the city ' s main hub of the city . The city is loc

# Сравнение с наивной версией (KV-кэш vs полный forward на каждом шаге)

In [15]:
import time

prompt = "The history of"
max_length = 100

torch.manual_seed(0)
start = time.perf_counter()
cached_text = generator.generate(prompt, tokenizer, max_length=max_length, temperature=0.8, top_k=50, top_p=0.9)
cached_time = time.perf_counter() - start

torch.manual_seed(0)
start = time.perf_counter()
naive_text = generator.lightning_module.generate(prompt, tokenizer, max_length=max_length, temperature=0.8, top_k=50, top_p=0.9)
naive_time = time.perf_counter() - start

print(f"С KV-кэшем ({cached_time:.2f}s):\n{cached_text}\n")
print(f"Без кэша, наивная версия ({naive_time:.2f}s):\n{naive_text}\n")
print(f"Ускорение: {naive_time / cached_time:.2f}x")

С KV-кэшем (0.74s):
The history of the city is unclear . It was reported in the 12th century that the city was under siege from the Battle of the Yellow Sea to the east of the River Tay , and a large portion of the River Tay to the south of the river . The town was replaced by the town of Tay , the first of three battalions of the state , and the first of the Battle of Britain . In the late

Без кэша, наивная версия (0.91s):
The history of the city is unclear . It was reported in the 12th century that the city was under siege from the Battle of the Yellow Sea to the east of the River Tay , and a large portion of the River Tay to the south of the river . The town was replaced by the town of Tay , the first of three battalions of the state , and the first of the Battle of Britain . In the late

Ускорение: 1.24x
